# 1. Core Pipeline — Fetch & Analyze

This notebook is the **producer** of the data assets the other two notebooks consume.

It does three things:

1. **Fetches** OHLC candle data from ByBit for a given symbol / interval / date range.
2. **Analyzes** every EMA in the configured range, counting touches, crosses, and per-direction test/held outcomes.
3. **Caches** both outputs to `data/` as parquet files, keyed by the configuration. Subsequent runs (here or in notebooks 2 & 3) load from cache instead of re-fetching.

After running this notebook, you can open `2_ema_analysis.ipynb` (interpretation / ranking) or `3_ema_backtesting.ipynb` (strategy backtest) — both will pick up the cached files automatically as long as the configuration matches.

__Goals:__
1. Build a ByBit Perpetual Futures EMA S/R Analyzer.
2. Quantify and compare EMA behavior to understand which periods price actually respects.

__Data:__ \
ByBit Perpetual Futures.

__Workflow:__ \
For each EMA period in a user-supplied range, every evaluated candle is placed in exactly one of five mutually-exclusive categories, so the categories always sum to evaluated_candles.

__Inputs:__
- symbol
- interval (ByBit codes: 1/3/5/15/30/60/120/240/360/720/D/W/M)
- start/end dates (YYYY-MM-DD)
- ema_range (any iterable of integers, e.g. range(1, 101) or [9, 21, 50, 200])
- delta
- delta_mode

__Delta modes:__
- "percent" — delta is a % of the EMA value at each candle (recommended, scales across price regimes) \
The allowed distance changes with price.
- "absolute" — delta is a fixed price distance in quote currency (e.g. USDT). Distance in price units. \
The allowed distance is always exactly the chosen 'number' regardless of price.

__Categories__ (mutually exclusive partition; every candle lands in one category):
- cross      → Low ≤ EMA ≤ High. The candle's range straddles the EMA — the level was *broken*, not held.
- low_touch  → Low > EMA AND Low − EMA ≤ delta. Entire candle above EMA; low approached EMA from above without crossing — a *test of support*.
- high_touch → High < EMA AND EMA − High ≤ delta. Entire candle below EMA; high approached EMA from below without crossing — a *test of resistance*.
- above      → Low > EMA AND Low − EMA > delta. Entire candle above EMA, low far away — clean uptrend, no test.
- below      → High < EMA AND EMA − High > delta. Entire candle below EMA, high far away — clean downtrend, no test.

__Invariants:__
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (the two are mutually exclusive — a candle either tests support OR resistance, never both)

__Notes:__
- The first period candles are skipped per EMA (warm-up); toggle with skip_warmup=False if you don't want that.
- Pagination handles arbitrarily long date ranges (1000-candle chunks).
- Uses ewm(span=period, adjust=False) — standard Wilder/TradingView-style EMA.


## Setup


In [1]:
from ema_core import load_or_fetch_klines, load_or_compute_result


## Configuration

Edit these to fetch a different market / range. Anything you change here also needs to match in notebooks 2 & 3 — they use the same values to find the cache files.

Configuration:
- interval = "60" for 1-hour candles
- ema_range  = range(5, 201, 5) calculates EMA 5, 10, 15, ..., 200
- delta_mode = "percent" or "absolute"
- delta %: the allowed distance between EMA and candle changes with price \
  delta=0.5 → "low within 0.5% of EMA"
- delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price. Distance in price units. \
  delta=40 → "low within $40 of EMA"

*Examples:*
- delta=0.10 is 0.10 % of the EMA 
- a 5 USDT price difference in % is calculated as:  5 / price (e.g. 67,000) × 100 = 0.00746%
- a 0.0075% delta in USDT is calculated as: price (e.g. 67,000) × 0.0075 / 100 = 5.025 USDT

In [ ]:
symbol     = "BTCUSDT"
interval   = "15"
start      = "2026-01-01"
end        = "2026-04-01"
ema_range  = range(1, 200, 1)
delta      = 20
delta_mode = "absolute"          # "percent" (of EMA) or "absolute" (quote ccy)
skip_warmup = True                # drop the first `period` candles per EMA
category    = "linear"            # "linear" (USDT perps) or "inverse" (coin-margined)


## Fetch OHLC

`load_or_fetch_klines` returns the cached parquet if present, otherwise fetches from ByBit and caches the result. \
Pass `force_refetch=True` to refresh.


Dataframe of raw OHLCV candles (klines) from ByBit:

In [ ]:
df = load_or_fetch_klines(symbol, interval, start, end, category)
print(f"  -> {len(df)} candles loaded, interval={interval}, {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
df.head()


## Analyze EMA touches

`load_or_compute_result` runs `analyze_ema_touches` over `ema_range` and caches the per-EMA counts. \
The cache key includes a hash of `(ema_range, delta, delta_mode, skip_warmup)` — change any of those and a fresh cache entry is created automatically.

skip_warmup : drop the first period candles per EMA \
skip_warmup=True: ignore the first N candles (where N = the EMA period) \
delta_mode  : "percent" (of EMA value) or "absolute" (fixed quote-currency) \
delta %: the allowed distance changes with price \
delta absolute: the allowed distance is always exactly the chosen 'number' regardless of price

Main algorithm:
1. Fetches ByBit perpetual futures data.
2. For every EMA period in ema_range:
   - Computes the EMA on close prices.
   - Assigns every evaluated candle to a category.
   - Categories are mutually exclusive.
3. Returns a clean DataFrame with one row per EMA period.


Partition table: every evaluated candle is placed in exactly one category.

| category     | condition                                                            | behavior | meaning |
|--------------|----------------------------------------------------------------------|---------|----------|
| cross      | Low ≤ EMA ≤ High | candle crosses EMA | level broken             |
| low_touch  | Low > EMA AND Low − EMA ≤ delta | entire candle above EMA, Low within delta of EMA, Low approached EMA from above | support held, no break |
| high_touch | High < EMA AND EMA − High ≤ delta | entire candle below EMA, High within delta of EMA, High approached EMA from below | resistance held, no break |
| above      | Low > EMA AND Low − EMA > delta | entire candle above EMA, Low further than delta | clean uptrend, no test of EMA |
| below      | High < EMA AND EMA − High > delta | entire candle below EMA, High further than delta | clean downtrend, no test of EMA |

Invariants:
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (mutually exclusive — a candle either tests support OR resistance, never both)


Dataframe of candle categories by EMA period:

In [ ]:
result = load_or_compute_result(df, symbol, interval, start, end, ema_range, delta, delta_mode, skip_warmup, category)
print(f"  -> {len(result)} EMA periods analyzed.")
result.head(10).style.hide(axis="index")


__Alternative: convenience wrapper__

Convenience wrapper:
- does NOT use the parquet cache
- for cached, use load_or_fetch_klines + load_or_compute_result instead
- returns 2 dataframes: df, result

In [17]:
from ema_core import run
df, result = run(symbol, interval, start, end, ema_range, delta, delta_mode)

Fetching BTCUSDT interval=15 from 2026-01-01 to 2026-04-01 ...
  -> 8641 candles downloaded


## Sanity check

A quick look at the partition counts for a handful of EMAs. \
The invariant `cross + low_touch + high_touch + above + below = evaluated_candles` should hold for every row.


In [5]:
sample = result.iloc[::max(1, len(result) // 10)][['ema', 'low_touch', 'high_touch', 'cross', 'above', 'below', 'evaluated_candles']]
sample.assign(_sum=sample[['low_touch','high_touch','cross','above','below']].sum(axis=1)).head(10)


,ema,low_touch,high_touch,cross,above,below,evaluated_candles,_sum
0,1,0,0,8640,0,0,8640,8640
19,20,234,313,2677,2615,2782,8621,8621
38,39,174,184,1813,3043,3388,8602,8602
57,58,133,162,1355,3247,3686,8583,8583
76,77,113,125,1136,3334,3856,8564,8564
95,96,98,115,1013,3347,3972,8545,8545
114,115,119,93,913,3331,4070,8526,8526
133,134,77,77,863,3312,4178,8507,8507
152,153,79,84,821,3243,4261,8488,8488
171,172,62,69,773,3214,4351,8469,8469


## Data export

Saving dataframes to CSV for further analysis in Excel or Tableau.

In [6]:
filename_df = f"df_{symbol}_{interval}_{start}_{end}.csv"
df.to_csv(filename_df, index=False)
print(f"\nResults saved to {filename_df}")


Results saved to df_BTCUSDT_15_2026-01-01_2026-04-01.csv


In [7]:
filename_ema = f"ema_analysis_{symbol}_{interval}_{start}_{end}_{ema_range}_{delta}.csv"
result.to_csv(filename_ema, index=False)
print(f"\nResults saved to {filename_ema}")


Results saved to ema_analysis_BTCUSDT_15_2026-01-01_2026-04-01_range(1, 200)_20.csv


## Conclusion

Both caches are now on disk under `data/`. \
Open `2_ema_analysis.ipynb` next to interpret which EMAs act as support / resistance / universal S/R, \
or skip ahead to `3_ema_backtesting.ipynb` to backtest a strategy on this dataset.
